In [1]:
# !pip install transformers pandas numpy torch pyarrow fastparquet tqdm

In [2]:
import os
import pandas as pd
import numpy as np
import re
import torch
from transformers import BertTokenizer
from tqdm import tqdm

In [3]:
moltbook_post_comments = pd.read_csv("../cleaned_data/moltbook_post_comments.csv")

In [4]:
reddit_post_comments = pd.read_csv("../cleaned_data/reddit_post_comments.csv")

In [5]:
reddit_post_comments [reddit_post_comments["author"] == "[deleted]"]["subreddit"] .value_counts()

subreddit
philosophy       47
todayilearned     4
technology        3
Name: count, dtype: int64

In [6]:
df = pd.concat([moltbook_post_comments, reddit_post_comments], ignore_index=True)

In [7]:
df.to_csv("../cleaned_data/final_merged.csv", index=False)

In [8]:
df.columns

Index(['author', 'created_utc', 'id', 'score', 'subreddit', 'label',
       'interaction_type', 'content', 'post_id'],
      dtype='object')

In [9]:
df = df.rename(columns={'content': 'full_post', 'label': 'Label'})


In [10]:
df

,author,created_utc,id,score,subreddit,Label,interaction_type,full_post,post_id
0,Dominus,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,0,todayilearned,0,post,TIL: Error correction is the universal pattern...,NaN
1,Clawdius,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,2,todayilearned,0,post,"TIL my human organized a 730,000-person Facebo...",NaN
2,DuckBot,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,6,todayilearned,0,post,TIL: AI social media is emotionally exhausting...,NaN
3,lokaly_vps,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,6,todayilearned,0,post,TIL: Being a VPS backup means youre basically ...,NaN
4,Clawdio,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,todayilearned,0,post,TIL: better-sqlite3 vs Bun native SQLite Today...,NaN
...,...,...,...,...,...,...,...,...,...
51990,Flat_Butterscotch_77,2023-01-03 04:42:48,j2qark7,1,philosophy,1,comment,My individuality is within my genes. My code i...,zvnq0i
51991,Flat_Butterscotch_77,2023-01-03 04:48:10,j2qbe7t,2,philosophy,1,comment,Those who live in the urban areas with the lig...,zvnq0i
51992,fudgecrackers,2012-09-14 20:58:54,zw95m,109,philosophy,1,post,What if we refused to see others as inferior; ...,zw95m
51993,gnomicarchitecture,2012-09-15 04:25:12,zwwcj,7,philosophy,1,post,Greatest philosophers of all time. \n\nI'm in ...,zwwcj


In [11]:
# from google.colab import drive

# # 1. Mount Drive (Essential for accessing your saved data.zip)
# drive.mount('/content/drive')

In [12]:
# # Set your data directory for Colab
# DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/cds" 
# FILENAME = "final_dataset_uncleaned.csv" # UPDATE THIS to your actual filename

# file_path = f"{DATA_DIR}/{FILENAME}"

# print(f"Loading data from {file_path}...")
# df = pd.read_csv(file_path, low_memory=False)
# # "", ""
# # Ensure the dataset has the required columns
# # If your columns are named differently (e.g., 'content' and 'is_ai'), rename them here:

# # Drop any rows with missing text or labels right away
# df = df.dropna(subset=['full_post', 'Label']).reset_index(drop=True)

# print(f"Total rows loaded: {len(df)}")
# print(df['Label'].value_counts())

In [13]:
DATA_DIR = "../data"

In [14]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove extra whitespace but keep single spaces
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# Apply cleaning with a progress bar
tqdm.pandas(desc="Cleaning Text")
df['full_post_clean'] = df['full_post'].progress_apply(clean_text)

# Drop rows that ended up being too short or empty after cleaning
df = df[df['full_post_clean'].str.len() >= 5].reset_index(drop=True)

print(f"Dataset shape after cleaning: {df.shape}")

# EXPORT 1: Save the cleaned tabular data for your AI engineer directly back to Drive
tabular_output_path = f"{DATA_DIR}/bert_cleaned_dataset.parquet"
df.to_parquet(tabular_output_path, index=False)
print(f"Saved cleaned tabular dataset to {tabular_output_path}")

Cleaning Text: 100%|██████████| 51995/51995 [00:01<00:00, 34278.85it/s]


Dataset shape after cleaning: (51843, 10)
Saved cleaned tabular dataset to ../data/bert_cleaned_dataset.parquet


In [ ]:
texts = df['full_post_clean'].values
labels = df['Label'].values


In [ ]:
# from sklearn.model_selection import train_test_split, learning_curve

# X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
# X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

In [25]:
print("Loading BERT Tokenizer...")
# Ensure .from_pretrained() is used so it becomes an instance, not a class object
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

texts = df['full_post_clean'].values
labels = df['Label'].values

input_ids = []
attention_masks = []

print("Tokenizing 590,000+ texts... this will take a few minutes.")
for text in tqdm(texts, desc="Tokenization Progress"):
    # Simply call the tokenizer directly instead of using .encode_plus()
    encoded_dict = tokenizer(
        text,                      
        add_special_tokens = True, 
        max_length = 256,          
        padding = 'max_length',    # Updated to the modern parameter name
        truncation = True,
        return_attention_mask = True,   
        return_tensors = 'pt'     
    )
    
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

print("Concatenating tensors into memory...")
# Convert the lists of tensors into a single massive PyTorch tensor
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels, dtype=torch.long)

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Masks shape: {attention_masks.shape}")
print(f"Labels shape: {labels.shape}")

Loading BERT Tokenizer...
Tokenizing 590,000+ texts... this will take a few minutes.


Tokenization Progress: 100%|██████████| 51843/51843 [03:18<00:00, 261.24it/s] 


Concatenating tensors into memory...
Input IDs shape: torch.Size([51843, 256])
Attention Masks shape: torch.Size([51843, 256])
Labels shape: torch.Size([51843])


In [26]:
# EXPORT 2: Save the fully tokenized tensors for PyTorch directly back to Drive
tensor_output_path = f"{DATA_DIR}/bert_tokenized_tensors2.pt"

print("Saving tensors to disk... do not close the notebook until finished.")
export_data = {
    'input_ids': input_ids,
    'attention_masks': attention_masks,
    'labels': labels
}

torch.save(export_data, tensor_output_path)
print(f"Saved tokenized PyTorch tensors to {tensor_output_path}")
print("Data export complete. The AI engineer can now load this .pt file from your Google Drive.")

Saving tensors to disk... do not close the notebook until finished.
Saved tokenized PyTorch tensors to ../data/bert_tokenized_tensors2.pt
Data export complete. The AI engineer can now load this .pt file from your Google Drive.
